# Step 1. Notebook Overview

**Purpose**

**What this notebook does**

* Converts text into numerical features

* Creates linguistic, statistical, and business-driven features

* Prepares final dataset for ML models

Feature engineering converts raw cleaned data into structured signals that models can learn from.

# Step 2. Import Required Libraries

**Purpose**

* Load only necessary libraries
* Keep notebook clean & reproducible

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

import matplotlib.pyplot as plt


##### I imported only essential libraries for text vectorization, encoding, and numerical feature creation.

# Step 3. Load Preprocessed Dataset

**Purpose**

* Load already cleaned data
* Avoid repeating preprocessing work
* Follow modular pipeline design

In [2]:
file_path = r"C:\Users\hp\Desktop\Fintech_Complaint_Analysis\data\processed\complaints_cleaned.csv"
df = pd.read_csv(file_path)

df.head()

,Date received,Product,Issue,Consumer complaint narrative,Company,text_length,cleaned_text,cleaned_text_length
0,2025-10-14,Credit reporting or other personal consumer re...,Incorrect information on your report,My credit report contains incorrect and mislea...,"EQUIFAX, INC.",57,credit report contains incorrect misleading item,48
1,2025-10-26,Credit reporting or other personal consumer re...,Incorrect information on your report,Be sure that disputed accounts are taken down ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",82,sure disputed account taken promptly accordanc...,51
2,2020-05-08,"Credit reporting, credit repair services, or o...",Incorrect information on your report,These are not my accounts.,Experian Information Solutions Inc.,26,account,7
3,2025-10-21,Credit reporting or other personal consumer re...,Incorrect information on your report,I do not recognize the aforementioned accounts...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",364,recognize aforementioned account collection ha...,253
4,2025-11-10,Credit reporting or other personal consumer re...,Incorrect information on your report,I am formally disputing inaccurate information...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",1599,formally disputing inaccurate information cred...,1150


##### I load the cleaned dataset directly to keep preprocessing and feature engineering decoupled.

# Step 4. Validate Required Columns

**Purpose :** 

1.Ensure necessary columns exist before feature creation
    
2.Prevent silent pipeline failures

In [3]:
required_cols = [
    'cleaned_text',
    'Consumer complaint narrative',
    'Product',
    'Issue',
    'Company',
    'Date received'
]

missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")
else:
    print("All required columns are present")


All required columns are present


##### I validate schema early to ensure robustness and avoid downstream errors.

# Step 5. Text Length Features

**Purpose :** 

1.Complaint length often correlates with severity and risk
    
2.Longer complaints → higher frustration → higher risk

In [4]:
df['char_length'] = df['Consumer complaint narrative'].astype(str).apply(len)
df['word_length'] = df['Consumer complaint narrative'].astype(str).apply(lambda x: len(x.split()))


##### Text length features help capture complaint intensity and customer effort.

# Step 6. Stopword Ratio Feature

**Purpose**

1.Measures how much of the complaint is meaningful content
    
2.Low ratio → more keywords → higher signal


In [5]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def stopword_ratio(text):
    words = text.split()
    if len(words) == 0:
        return 0
    stopword_count = sum(1 for w in words if w in stop_words)
    return stopword_count / len(words)

df['stopword_ratio'] = df['cleaned_text'].apply(stopword_ratio)


##### Stopword ratio helps identify complaints rich in meaningful keywords.

# Step 7. Numeric Keyword Risk Flags

**Purpose :**

1.Domain-specific risk indicators

2.Mimics real fintech risk logic

In [6]:
risk_keywords = [
    'fraud', 'scam', 'chargeback', 'unauthorized',
    'dispute', 'stolen', 'blocked', 'error',
    'failed', 'refund', 'denied'
]

def keyword_flag(text):
    return int(any(word in text for word in risk_keywords))

df['risk_keyword_flag'] = df['cleaned_text'].apply(keyword_flag)


##### I created domain-driven keyword flags to capture explicit risk signals.

# Step 8. TF-IDF Text Vectorization

**Purpose :**

1.Convert text into numerical vectors

2.Capture word importance across complaints

3.Industry-standard NLP approach


In [7]:
tfidf = TfidfVectorizer(
    max_features=300,
    ngram_range=(1, 2),
    min_df=5
)

tfidf_features = tfidf.fit_transform(df['cleaned_text'])

tfidf_df = pd.DataFrame(
    tfidf_features.toarray(),
    columns=tfidf.get_feature_names_out()
)


##### TF-IDF captures both word frequency and importance across complaints.

In [8]:
import joblib

joblib.dump(
    tfidf,
    r"C:\Users\hp\Desktop\Fintech_Complaint_Analysis\models\tfidf_vectorizer.pkl"
)
print("TF-IDF Vectorizer saved successfully")

TF-IDF Vectorizer saved successfully


# Step 9. Encode Categorical Features

**Purpose :**

1.ML models require numeric input

2.Encode product, issue, and company info

In [9]:
label_encoder = LabelEncoder()

df['product_encoded'] = label_encoder.fit_transform(df['Product'].astype(str))
df['issue_encoded'] = label_encoder.fit_transform(df['Issue'].astype(str))


##### Categorical encoding enables models to learn product- and issue-level risk patterns.

# Step 10. Time-Based Features

**Purpose :** 

1.Capture seasonal or operational trends

2.Useful for monitoring complaint spikes

In [10]:
df['Date received'] = pd.to_datetime(df['Date received'], errors='coerce')

df['complaint_year'] = df['Date received'].dt.year
df['complaint_month'] = df['Date received'].dt.month


##### Time features help identify trends and complaint surges.

# Step 11. Combine All Features

**Purpose :**

1.Merge text, numeric, categorical, and risk features
    
2.Create final modeling dataset

In [11]:
df_final = pd.concat(
    [
        df[
            [
                'char_length',
                'word_length',
                'stopword_ratio',
                'risk_keyword_flag',
                'product_encoded',
                'issue_encoded',
                'complaint_year',
                'complaint_month'
            ]
        ].reset_index(drop=True),
        tfidf_df.reset_index(drop=True)
    ],
    axis=1
)

df_final.head()


,char_length,word_length,stopword_ratio,risk_keyword_flag,product_encoded,issue_encoded,complaint_year,complaint_month,accordance,account,...,xxxx balance,xxxx date,xxxx xxxx,xxxx xxxxxxxx,xxxxxxxx,xxxxxxxx balance,xxxxxxxx xxxx,xxxxyear,year,yet
0,57,8,0.0,0,6,32,2025,10,0.000000,0.000000,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0
1,82,14,0.0,1,6,32,2025,10,0.555462,0.200713,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0
2,26,5,0.0,0,7,32,2020,5,0.000000,1.000000,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0
3,364,54,0.0,1,6,32,2025,10,0.000000,0.169315,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0
4,1599,233,0.0,1,6,32,2025,11,0.000000,0.112413,...,0.0,0.0,0.409834,0.040575,0.032309,0.0,0.039342,0.0,0.0,0.0


##### I combined textual, linguistic, categorical, and business features into a unified dataset.

# Step 12. Save Feature-Engineered Dataset

**Purpose :**

1.Persist features for modeling

2.Ensure reproducibility

In [12]:
feature_path = r"C:\Users\hp\Desktop\Fintech_Complaint_Analysis\data\processed\complaints_features.csv"

df_final.to_csv(feature_path, index=False)

print("Feature-engineered dataset saved successfully")


Feature-engineered dataset saved successfully


##### I saved the feature-engineered dataset separately to support clean model training.

# Step 13. Final Summary

**What i achieved**

* Text → numerical vectors

* Business risk signals

* Categorical encoding

* Time-based insights


This feature engineering pipeline transforms raw complaints into structured signals suitable for risk scoring models.